# Classification accuracy v2 — targeted experiments

The Phase-1 bake-off (`classification_bakeoff.ipynb`) and its companion
`classification_improvements.ipynb` already ran the obvious accuracy experiments, with
saved outputs, and **model swaps are a closed question**:

| Tested | Result | Verdict |
|---|---|---|
| `bge-reranker-base` vs `ms-marco-MiniLM-L-6-v2` | MRR 0.556 vs 0.545, but **807s vs 82s** (~10x) | Not worth it |
| `bge-large` / `bge-small` vs `bge-base` embeddings | 0.526 / 0.510 vs **0.529** | `bge-base` already best |
| BGE query-instruction prefix | Every model scored *higher without it* | Already shipped |
| `name + tactics + description` corpus text | 0.508 vs **0.529** for `name + description` | Tactic slugs hurt |

So this notebook targets two levers that are genuinely **untested**:

1. **Retrieval N for Method D.** `HYBRID_RETRIEVAL_N = 30` was chosen from a recall
   curve (recall@30 = 0.905 on the full 259-example set); ablation 5 in the
   improvements notebook only swept N for Method C (hybrid LLM), never for D
   (retrieve + rerank), which is what's actually in production.
2. **Corpus enrichment.** This session's corpus regeneration added each technique's
   MITRE mitigation text (`app/classification/corpus.py`'s `mitigation` field, via a
   live re-fetch of the STIX bundle — see `research/lib/attack_data.py`). Does
   appending it to the reranker's candidate text help or hurt matching?

**Methodology note:** run as a standalone script rather than executed inline in this
notebook (this project's established pattern — see
`classification_confidence_calibration.ipynb`), on the bake-off's own stratified
80-example sample (`seed=42`), matching its headline-comparison methodology. The code
below reproduces the exact logic that was run; the **Findings** section at the bottom
has the real measured numbers.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "..")
sys.path.insert(0, ".")

import numpy as np
from fastembed import TextEmbedding

from lib import attack_data, eval_data, metrics, rerank

DATA = Path("data")
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"
MAX_N = 50
SAMPLE_N = 80
SEED = 42

## Load corpus + eval set, embed once

Retrieval only needs to run once at the largest N (50); every smaller N in the sweep
just truncates the same ranked candidate list before reranking, so this doesn't need
to re-embed per N value.

In [ ]:
techniques = attack_data.load_cached(DATA / "attack_ics_techniques.json")
technique_by_id = {t.id: t for t in techniques}

examples = eval_data.load_eval_set(
    DATA / "procedure_examples.jsonl",
    DATA / "labeled_dataset.jsonl",
    DATA / "labeled_dataset.local.jsonl",
)
eval_data.validate_technique_ids(examples, set(technique_by_id))
sample = eval_data.sample_by_technique(examples, SAMPLE_N, seed=SEED)
print(f"Loaded {len(examples)} examples total, sampled {len(sample)} (seed={SEED}).")

In [ ]:
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

corpus_ids = [t.id for t in techniques]
corpus_texts_desc = [f"{t.name}. {t.description}" for t in techniques]
corpus_vecs = np.array(list(embedder.embed(corpus_texts_desc)))
corpus_vecs = corpus_vecs / np.linalg.norm(corpus_vecs, axis=1, keepdims=True)

query_texts = [ex.behavior_text for ex in sample]
query_vecs = np.array(list(embedder.embed(query_texts)))
query_vecs = query_vecs / np.linalg.norm(query_vecs, axis=1, keepdims=True)

retrieved_top50 = []
for qv in query_vecs:
    sims = corpus_vecs @ qv
    order = np.argsort(-sims)[:MAX_N]
    retrieved_top50.append([corpus_ids[i] for i in order])

recall_at_30 = np.mean([
    len(set(ex.true_technique_ids) & set(cands[:30])) / len(ex.true_technique_ids)
    for ex, cands in zip(sample, retrieved_top50, strict=True)
])
print(f"recall@30 on this 80-sample: {recall_at_30:.3f} (bake-off full-259: 0.905)")

## Sanity check: does this harness reproduce the bake-off's own numbers?

Same config as production (`app/classification/retrieval.py` /
`app/classification/reranking.py`): N=30, no query-instruction prefix, reranker
candidate text is `f"{name}. {description_full}"`.

In [ ]:
reranker = rerank.CrossEncoderReranker(model_name=RERANKER_MODEL)


def rerank_at_n(n, candidate_text_fn):
    results = []
    for ex, cands, qtext in zip(sample, retrieved_top50, query_texts, strict=True):
        pool = cands[:n]
        candidates = [(tid, candidate_text_fn(technique_by_id[tid])) for tid in pool]
        ranked = reranker.rerank(qtext, candidates)
        results.append(metrics.MethodResult(
            report_id=ex.id,
            true_ids=ex.true_technique_ids,
            ranked_ids=[r.technique_id for r in ranked],
        ))
    return metrics.rank_score_method(f"D (N={n})", results)


def desc_only(t):
    return f"{t.name}. {t.description_full}"


def desc_plus_mitigation(t):
    base = f"{t.name}. {t.description_full}"
    return f"{base} {t.mitigation}" if t.mitigation else base


baseline = rerank_at_n(30, desc_only)
print(f"MRR={baseline.mrr:.3f}  MAP={baseline.map:.3f}  R@3={baseline.recall_at_k[3]:.3f}")
print("(bake-off's own 80-sample headline for Method D: MRR=0.559, R@3=0.631)")

## Experiment 1 — retrieval N sweep

In [ ]:
n_results = {30: baseline}
for n in (20, 40, 50):
    n_results[n] = rerank_at_n(n, desc_only)

for n in sorted(n_results):
    r = n_results[n]
    print(f"N={n:2}  MRR={r.mrr:.3f}  MAP={r.map:.3f}  R@3={r.recall_at_k[3]:.3f}")

## Experiment 2 — append mitigation text to reranker candidate text

In [ ]:
with_mitigation = rerank_at_n(30, desc_plus_mitigation)
print(f"desc only:        MRR={baseline.mrr:.3f}  MAP={baseline.map:.3f}")
print(f"desc + mitigation: MRR={with_mitigation.mrr:.3f}  MAP={with_mitigation.map:.3f}")

## Findings (real numbers, from the standalone run)

**Sanity check:** MRR=0.533, MAP=0.530, R@1=0.406, R@3=0.606, R@5=0.681 on this
80-sample — close to but not identical to the bake-off's own reported MRR=0.559 /
R@3=0.631 for the same nominal config. The gap is most likely corpus drift: this
session re-fetched the live STIX bundle to add mitigation text (see the corpus
enrichment work), and MITRE revises technique descriptions between fetches — the
underlying reference data is a moving target, not a fixed historical artifact. All
comparisons *within* this run use the same freshly-fetched corpus throughout, so the
relative comparisons below remain valid even though the absolute baseline shifted
slightly from the original bake-off's numbers.

**Experiment 1 — retrieval N sweep (adopted):**

| N | MRR | MAP | R@3 |
|---|---|---|---|
| 20 | 0.521 | 0.518 | 0.581 |
| 30 (previous default) | 0.533 | 0.530 | 0.606 |
| 40 | 0.550 | 0.546 | 0.619 |
| **50** | **0.559** | **0.556** | **0.619** |

Monotonic across every N tested — a real, low-noise signal, not a lucky data point.
N=50 beats N=30 by +0.026 MRR (+4.9% relative) and +0.013 R@3 (+2.1% relative), for a
reranking latency cost that scales roughly linearly with N (~1.02s/example at N=30 vs
~1.64s/example at N=50 on this machine, measured across the 80-sample runs above).
**Adopted**: `HYBRID_RETRIEVAL_N` raised from 30 to 50 in
`app/classification/retrieval.py`. Recall@50 was already known to reach 0.967 on the
full 259-example set (vs 0.905 at 30), so this mostly recovers previously-unreachable
true techniques that were simply never in the reranker's candidate pool at N=30.

**Caveat carried forward, not resolved this session:** `_MIN_TOP1_MARGIN = 1.25`
(`app/classification/service.py`) was calibrated against N=30's candidate pool. A wider
pool can occasionally let a marginal candidate outscore what would have been rank-2 at
N=30, which could shift the margin distribution slightly. Re-deriving it means
re-running `classification_confidence_calibration.ipynb`'s full-259 pass (~8 minutes)
against the new N — flagged as a known follow-up, not done here, since the threshold's
own documented confidence was already "not a rigorously cross-validated cutoff" even
before this change.

**Experiment 2 — mitigation text appended to reranker candidates (rejected):**

| Config | MRR | MAP |
|---|---|---|
| description only (current) | 0.533 | 0.530 |
| description + mitigation | 0.506 | 0.504 |

Confirms the hypothesis stated in `research/lib/attack_data.py`'s module docstring:
mitigation text describes the *fix*, not the *attack*, so it's a worse semantic match
for attacker-behaviour queries than description text alone — a real, measured
regression (-0.027 MRR), not just a theoretical concern. **Rejected.** Mitigation text
stays display-only (`TechniqueRollup.mitigation`), never fed into retrieval or
reranking, exactly as already implemented in `app/classification/service.py`.

## Deferred: tactic as a retrieval prior/filter

A different idea from concatenating tactic slugs into corpus text (already tested and
shown to hurt, MRR 0.508 vs 0.529 — see `classification_improvements.ipynb` ablation
2): using tactics as a *gate or boost* — e.g. inferring a likely tactic and biasing
retrieval toward techniques sharing it.

**Not implemented.** The missing piece isn't the gating logic, it's a trustworthy way
to independently predict a report's likely tactic *before* running retrieval — without
one, "infer the tactic" just becomes another classification problem with its own error
rate, potentially compounding rather than reducing mistakes. Recording this as a
deliberately deferred idea rather than forcing a weak version of it just to have
attempted it.